In [1]:
%%capture
!pip install unsloth

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
import glob
from unsloth import FastLanguageModel
import torch

OUTPUT_GGUF_DIR = "/content/drive/My Drive/gguf-model"
OUTPUT_CHECKPOINTS_DIR = "/content/drive/My Drive/Lab2-checkpoints"

os.makedirs(OUTPUT_GGUF_DIR, exist_ok=True)
os.makedirs(OUTPUT_CHECKPOINTS_DIR, exist_ok=True)

max_seq_length = 1024
dtype = None  # None for auto detection.
load_in_4bit = True  # Use 4bit quantization to reduce memory usage.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [12]:
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from datasets import load_dataset, concatenate_datasets

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        )
        for convo in convos
    ]
    return {"text": texts}

# fintome = load_dataset("mlabonne/FineTome-100k", split="train")

# fintome = standardize_sharegpt(fintome)

ultra = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft",
)

# Map `messages` -> `conversations` so it matches what formatting_prompts_func expects
def ultrachat_to_conversations(examples):
    return {"conversations": examples["messages"]}

ultra = ultra.map(
    ultrachat_to_conversations,
    batched=True,
    remove_columns=ultra.column_names,  # keep only "conversations"
)

# dataset = concatenate_datasets([fintome, ultra])
dataset = ultra

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/207865 [00:00<?, ? examples/s]

In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 1,
    warmup_steps = 5,
    num_train_epochs = 1,
    # max_steps = 10,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 100000,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = OUTPUT_CHECKPOINTS_DIR,
    save_strategy = "steps",   # Save by steps
    save_steps = 100,
    save_total_limit = 1,      # Keep last N checkpoints to save space
    logging_dir = os.path.join(OUTPUT_CHECKPOINTS_DIR, "logs"),
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/207865 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/207865 [00:00<?, ? examples/s]

In [ ]:
checkpoint_pattern = os.path.join(OUTPUT_CHECKPOINTS_DIR, "checkpoint-*")
checkpoint_dirs = glob.glob(checkpoint_pattern)

resume_from_checkpoint = None
if checkpoint_dirs:
    # sort by step number: checkpoint-1000, checkpoint-2000, ...
    def _step_num(path):
        try:
            return int(path.split("-")[-1])
        except ValueError:
            return -1

    checkpoint_dirs = sorted(checkpoint_dirs, key=_step_num)
    resume_from_checkpoint = checkpoint_dirs[-1]
    print(f"Found checkpoint: {resume_from_checkpoint}. Resuming training from it.")
else:
    print("No checkpoints found. Starting training from scratch.")

trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

The model is already on multiple devices. Skipping the move to device specified in `args`.


No checkpoints found. Starting training from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 207,865 | Num Epochs = 1 | Total steps = 25,984
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

In [ ]:
from datetime import datetime

ts = datetime.now().strftime("%m%d%H%M")

In [ ]:
OUTPUT_GGUF_DIR = "/content/drive/MyDrive/gguf-model"
MERGED_MODEL_FILE_PATH = "merged_model"
GGUF_OUTPUT_FILE_PATH = f"{OUTPUT_GGUF_DIR}/model-1B-fine-tuned-{ts}.gguf"

In [ ]:
model.save_pretrained_merged(MERGED_MODEL_FILE_PATH, tokenizer, save_method = "merged_16bit")
# tokenizer.save_pretrained(MERGED_MODEL_FILE_PATH)

In [ ]:
import os

if not os.path.isdir("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp.git
else:
    print("llama.cpp already present.")
!pip install -U "transformers" "huggingface_hub"
!cd llama.cpp && make -s

#  GGUF conversion
!python llama.cpp/convert_hf_to_gguf.py {MERGED_MODEL_FILE_PATH} --outfile {GGUF_OUTPUT_FILE_PATH} --outtype q8_0